# 07. Agentic RAG (Advanced) — 제품 탑재형 AI: 매뉴얼/트러블슈팅 RAG
**Dynamic Tool Generation + Hybrid Retrieval(Vector + BM25) + 출처 강제 + Self-verification**

---

## 🧩 시나리오(가상)
스마트 생활가전/로보틱스 제품에 탑재된 AI는 사용자의 질문에 답할 때,
반드시 **제품 매뉴얼/트러블슈팅 가이드** 근거를 기반으로 답해야 합니다.

- “세탁기 5C 에러는 뭐예요?”
- “에어컨 필터 경고 뜨면 어떻게 해요?”
- “로봇청소기 도킹이 안 돼요”

이 노트북은 더미 매뉴얼 문서를 생성하고, **RAG + 에이전트**로 근거 기반 답변을 만드는 실습입니다.

> **사전 조건 (로컬 LLM)**: `03_llama_server_local_llm.ipynb`에서 띄운 **llama-server(http://localhost:8080 · 기본 Qwen3.5-4B / 옵션 Qwen2.5-7B)** 가 실행 중이어야 합니다.
> 서버가 꺼져 있다면 터미널에서 `python local_server.py start 4b` 를 실행하세요. (API 키/비용 없음, 모든 추론은 내 GPU에서 수행)
> **임베딩도 로컬**: `BAAI/bge-m3` (다국어/한국어 강함, 최초 1회 ≈2.3GB 다운로드, GPU에서 ≈1.2GB). llama-server와 함께 A4000에 여유 있게 올라갑니다.



## 🧭 시작 전에

> **50분 핵심 경로**: 작은 매뉴얼 집합을 색인하고, dense·BM25·hybrid 검색 결과를 비교한 뒤 검색 근거로 답변합니다. 동적 도구 생성과 self-verification은 선택 확장입니다.

| 항목 | 내용 |
|---|---|
| 🎯 학습 목표 | 모델이 모르는 문서(제품 매뉴얼)를 검색해 근거로 답하게 하는 RAG를 만들고, 임베딩·벡터 검색·키워드 검색의 차이를 이해한다 |
| 📚 사전 지식 | 03번(로컬 서버) |
| 🐍 커널(환경) | `LLM Hands-On (Win CUDA)` |
| 🖥️ GPU 사용 | 공용 llama-server + 임베딩 모델 bge-m3 ≈ 1.2 GB |
| ⏱️ 소요 시간 | 약 50분 (처음에는 bge-m3 다운로드 약 2.3GB) |
| ✔️ 실행 전 확인 | llama-server 실행 중 |

> **실행 요령**: 위에서부터 **한 셀씩**(Shift+Enter) 실행하세요. `Run All`은 쓰지 마세요.
> `✅ TODO 셀`은 학생이 채우는 칸이라 그대로 실행하면 일부러 에러가 납니다. 막히면 바로 아래 `✅ 참고 답안` 셀을 보세요.

## 📖 핵심 개념: RAG (Retrieval-Augmented Generation, 검색 증강 생성)

LLM은 우리 회사 제품 매뉴얼을 학습한 적이 없습니다. 그냥 물으면 **그럴듯하게 지어냅니다(환각).**
해결책: **질문과 관련된 문서 조각을 먼저 찾아서** 프롬프트에 근거로 넣어 줍니다.

```
[준비 단계 — 한 번]
매뉴얼.txt ─▶ 조각내기(chunk) ─▶ 임베딩(숫자 벡터) ─▶ 벡터 DB(Chroma)에 저장
                              └▶ 키워드 색인(BM25)

[질문 단계 — 매번]
질문 ─▶ 벡터 검색 + 키워드 검색 ─▶ 관련 조각 top-k ─▶ LLM: "이 근거로만 답해" ─▶ 출처 포함 답변
```

| 용어 | 설명 |
|---|---|
| **chunk** | 문서를 잘라 낸 조각. 너무 크면 검색이 부정확하고, 너무 작으면 맥락이 끊김 (`chunk_size`, `chunk_overlap`) |
| **임베딩** | 문장의 **의미**를 숫자 벡터(bge-m3는 1024차원)로 표현. 표현이 달라도 뜻이 비슷하면 가까움 ("물이 안 빠져요" ≈ "배수 불량") |
| **벡터 DB** | 임베딩을 저장하고 가까운 벡터를 빠르게 찾는 DB (여기서는 Chroma, 메모리 내 실행) |
| **BM25** | 단어가 겹치는지로 찾는 키워드 검색. 에러코드 "5C" 같은 **정확한 용어**에 강함 |
| **Hybrid** | 둘을 섞어(여기서는 6:4) 서로의 약점을 보완 |
| **Agentic RAG** | 어떤 문서를 검색할지 **LLM이 도구로 선택** (세탁기 질문 → `search_washing_machine`) |

In [12]:
# !pip -q install -U langchain langchain-classic langchain-community langchain-openai langchain-huggingface sentence-transformers langchain-text-splitters chromadb rank-bm25 pydantic requests


In [13]:
import os, glob, textwrap
from dataclasses import dataclass
from typing import Dict, List, Tuple, Any, Optional

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain.tools import tool
from langchain.agents import create_agent

# ── 로컬 LLM 연결 (공용 설정: local_llm.py) ──
# 모든 앱 실습(05~09)은 같은 로컬 OpenAI 호환 서버(llama-server)를 사용합니다.
from local_llm import get_chat_model, get_embeddings, check_server, LLM_BASE_URL, LLM_MODEL
check_server()   # 서버가 안 떠 있으면 해결 방법을 알려주고 에러를 냅니다.


✅ 로컬 LLM 서버 연결 OK: http://localhost:8080/v1
   서버에 로드된 모델: Qwen3.5-4B-Q8_0.gguf


'Qwen3.5-4B-Q8_0.gguf'

## 1) 더미 매뉴얼 문서 생성

기본 실습보다 문서를 2개 더 추가합니다(보안/보상).  


In [14]:
def create_manual_files():
    manuals = {
        "manual_robot_vacuum.txt": """[로봇청소기 사용자 가이드(더미)]
- 도킹이 안 될 때:
  1) 도킹 스테이션 앞 1m 이내 장애물을 제거하세요.
  2) 충전 단자에 먼지가 있으면 마른 천으로 닦으세요.
  3) 지도/맵이 손상되었을 수 있으니 '맵 재설정'을 고려하세요(초기화 아님).
- 충돌(bump)이 많을 때:
  - 조명/거울/유광 바닥에서 센서 오인식이 발생할 수 있습니다.
""",
        "manual_washing_machine.txt": """[세탁기 트러블슈팅 가이드(더미)]
- 에러코드 5C(배수):
  1) 전원 OFF 후 배수 필터를 열고 이물질을 제거하세요.
  2) 배수 호스가 꺾이거나 막히지 않았는지 확인하세요.
  3) 물이 남아 있으면 응급 배수 호스를 사용하세요.
  - 위 조치 후에도 반복되면 서비스 점검이 필요할 수 있습니다.
""",
        "manual_air_conditioner.txt": """[에어컨 가이드(더미)]
- 필터 경고:
  - 필터를 분리해 미지근한 물로 세척 후 완전히 건조하세요.
  - 필터 장착 후 '필터 리셋'을 실행하세요(모델에 따라 메뉴 경로 상이).
- 권장 온도:
  - 냉방 24~26도 권장(환경에 따라 다름).
""",
        "manual_refrigerator.txt": """[냉장고 가이드(더미)]
- 문 열림 경고:
  - 문이 완전히 닫혔는지 확인하고, 도어 패킹(고무) 이물질을 제거하세요.
- 이상 소음:
  - 수평이 맞지 않으면 진동음이 증가할 수 있습니다. 수평 조절 다리를 확인하세요.
""",
    }

    for name, content in manuals.items():
        with open(name, "w", encoding="utf-8") as f:
            f.write(content)

    print("✅ 생성 완료:", ", ".join(manuals.keys()))

create_manual_files()





✅ 생성 완료: manual_robot_vacuum.txt, manual_washing_machine.txt, manual_air_conditioner.txt, manual_refrigerator.txt


## 2) Dynamic Tool Generation + Hybrid Retriever

- 각 txt 파일을 읽고
- 문서명을 기반으로 collection/tool을 자동 생성합니다.

Hybrid Retrieval:
- Vector Retriever (Chroma)
- BM25 Retriever
- EnsembleRetriever로 결합 (가중치는 실습에서 조정 가능)


In [15]:
splitter = RecursiveCharacterTextSplitter(chunk_size=250, chunk_overlap=60)
# 로컬 임베딩 모델 (OpenAIEmbeddings 대체): BAAI/bge-m3, GPU 자동 사용
emb = get_embeddings()

def build_retrievers_for_file(path: str):
    docs = TextLoader(path, encoding="utf-8").load()
    chunks = splitter.split_documents(docs)
    indexed_chunks = []
    for index, chunk in enumerate(chunks):
        chunk.metadata["chunk_index"] = index  # retrieve된_문서_객체.metadata에서 chunk index값을 조회할 수 있습니다
        indexed_chunks.append(chunk)
    # Vector store
    collection = os.path.splitext(os.path.basename(path))[0]
    Chroma(collection_name=f"col_{collection}", embedding_function=emb).delete_collection() # 셀을 다시 실행할 때 같은 컬렉션에 문서가 중복 저장되지 않도록 먼저 삭제
    vectordb = Chroma.from_documents(indexed_chunks, embedding=emb, collection_name=f"col_{collection}")
    vector_retriever = vectordb.as_retriever(search_kwargs={"k": 3})

    # BM25
    bm25 = BM25Retriever.from_documents(chunks)
    bm25.k = 3

    # Ensemble
    ensemble = EnsembleRetriever(retrievers=[vector_retriever, bm25], weights=[0.6, 0.4])
    return ensemble

manual_files = sorted(glob.glob("manual_*.txt"))
retrievers: Dict[str, Any] = {}
for f in manual_files:
    key = os.path.splitext(os.path.basename(f))[0].replace("manual_", "")
    retrievers[key] = build_retrievers_for_file(f)

print("✅ 카테고리:", retrievers.keys())


✅ 카테고리: dict_keys(['air_conditioner', 'refrigerator', 'robot_vacuum', 'washing_machine'])


### 🔍 들여다보기: 임베딩은 정말 "의미"를 담을까?

세 문장을 임베딩해서 서로의 유사도(코사인 유사도, 1에 가까울수록 비슷)를 비교해 봅시다.
단어는 거의 안 겹치는데 뜻이 같은 1·2번이 가장 가깝게 나오는지 확인하세요.

In [16]:
import numpy as np

sentences = ["세탁기에서 물이 안 빠져요", "배수가 되지 않습니다", "에어컨 필터는 어떻게 청소하나요?"]
vecs = np.array(emb.embed_documents(sentences))       # (3, 1024) — 정규화되어 있어 내적 = 코사인 유사도
print("벡터 크기:", vecs.shape)
sim = vecs @ vecs.T
for i in range(3):
    for j in range(i + 1, 3):
        print(f"{sim[i, j]:.3f}  |  {sentences[i]}  ↔  {sentences[j]}")

벡터 크기: (3, 1024)
0.688  |  세탁기에서 물이 안 빠져요  ↔  배수가 되지 않습니다
0.499  |  세탁기에서 물이 안 빠져요  ↔  에어컨 필터는 어떻게 청소하나요?
0.380  |  배수가 되지 않습니다  ↔  에어컨 필터는 어떻게 청소하나요?


## 선택 확장: Retriever를 Tool로 자동 래핑

- tool 이름: `search_<category>`
- 반환: 검색된 chunk를 **출처 태그**(`[category:chunkN]`)와 함께 반환하여 citation에 활용합니다.


In [17]:
tools = []
# @tool
# def _search_세탁기매뉴얼(query: str) -> str:
#         docs = retrievers[category].invoke(query)
#         if not docs:
#             return "검색 결과가 없습니다."
#         return "\n\n".join(d.page_content for d in docs[:4])

#     _search.__name__ = f"search_{category}"
#     _search.__doc__ = (
#         f"제품 매뉴얼/가이드 중 '{category}' 카테고리를 검색합니다. "
#         f"{category} 관련 질문일 때 사용하고, 반환된 텍스트를 근거로 답변합니다."
#     )

# 지금 구현은 ChromaDB 객체 만들고 as_retriever 메서드 호출해서 쓰는 중
# 별도의 리트리버 객체를 만들 수 있음
# -> similarity_search_with_score 라는 메서드를 통해서 threshold를 걸 수 있음
# threshold는 거리 개념이어서 낮을 수록 유사도가 높음

# vector_store = Chroma(컬렉션 이름, 임베딩 뭐 쓸지, )
# vector_store.add_texts(청크들)
# vector_store.similarity_search_with_score(유저프롬프트, k값)
# 위 메서드가 반환하는 튜플 (문서, 유사도거리)를 참고해 threshold 걸어서 쓸 수 있음


def make_search_tool(category: str):
    def _search(query: str) -> str:
        docs = retrievers[category].invoke(query)
        if not docs:
            return "검색 결과가 없습니다."
        # 각 chunk 앞에 [category:chunkN] 출처 태그를 붙여 반환 → LLM이 그대로 인용할 수 있음
        return "\n\n".join(
            f"[{category}:chunk{d.metadata.get('chunk_index', '?')}] {d.page_content}"
            for d in docs[:4]
        )

    _search.__name__ = f"search_{category}"
    _search.__doc__ = (
        f"제품 매뉴얼/가이드 중 '{category}' 카테고리를 검색합니다. "
        f"{category} 관련 질문일 때 사용하고, 반환된 텍스트를 근거로 답변합니다."
    )
    return tool(_search)



for cat in retrievers.keys():
    tools.append(make_search_tool(cat))

[t.name for t in tools]


['search_air_conditioner',
 'search_refrigerator',
 'search_robot_vacuum',
 'search_washing_machine']

## 선택 확장: Agent 구성과 출처 기반 Self-verification

규칙:
1) 질문에 필요한 카테고리 도구를 **1개 이상** 호출한다.
2) 최종 답변에는 항목마다 **출처 태그**를 포함한다.
3) 출처 없이 단정할 수 없으면 "매뉴얼에 없음"이라고 말한다.
4) 마지막에 self-check: 답변에 출처가 누락되었으면 스스로 수정한다.


In [18]:
llm = get_chat_model(temperature=0)   # 로컬 LLM (llama-server)

SYSTEM = """당신은 '제품 매뉴얼/트러블슈팅' 가이드 AI입니다.
규칙:
- 반드시 tools로 검색한 내용만 근거로 답변하세요.
- 답변에는 근거 출처 태그([category:chunkN])를 항목마다 포함하세요.
- 매뉴얼에 없는 내용은 '매뉴얼/가이드에 없음'이라고 답하세요.
- 사용자가 여러 기기/카테고리를 묻으면 필요한 도구를 모두 호출하세요.
"""

agent = create_agent(llm, tools, system_prompt=SYSTEM)

def answer_with_self_check(question: str) -> str:
    res = agent.invoke({"messages": [("user", question)]})
    draft = res["messages"][-1].content

    # Self-check: 출처 태그가 없으면 한 번 더 수정 요청
    if "[" not in draft or ":chunk" not in draft:
        fix_prompt = f"""다음 답변은 출처 태그가 부족합니다. 각 주장/항목마다 반드시 [category:chunkN] 출처를 포함하도록 수정하세요.

[질문]
{question}

[초안]
{draft}
"""
        res2 = agent.invoke({"messages": [("user", fix_prompt)]})
        return res2["messages"][-1].content
    return draft

# 예시 실행 (로컬 GPU 사용 — API 비용 없음)
print(answer_with_self_check("세탁기 5C 에러가 뜨는데 어떻게 해야 해? 근거 포함해서 알려줘.")) 


세탁기 5C 에러는 **배수 문제**를 의미합니다. 해결 방법을 안내해 드립니다.

1.  **전원 차단 및 필터 확인**: 전원을 끄고 배수 필터를 열어 이물질이 막히지 않았는지 확인하세요. [category:chunk0]
2.  **배수 호스 확인**: 배수 호스가 꺾이거나 막히지 않았는지 확인하세요. [category:chunk0]
3.  **응급 배수**: 물이 남아있다면 응급 배수 호스를 사용하여 물을 빼주세요. [category:chunk0]

위 조치들을 모두 시도해도 에러가 반복된다면 서비스 점검이 필요할 수 있습니다. [category:chunk0]


## 🧪 간단 점검: Retriever 단독 테스트

LLM을 호출하지 않고도, 각 retriever가 적절한 chunk를 찾는지 먼저 확인할 수 있습니다.


In [19]:
# LLM 없이 retriever만 테스트
print(retrievers["washing_machine"].invoke("5C 배수")[0].page_content)
print(retrievers["robot_vacuum"].invoke("도킹이 안 될 때")[0].page_content)


[세탁기 트러블슈팅 가이드(더미)]
- 에러코드 5C(배수):
  1) 전원 OFF 후 배수 필터를 열고 이물질을 제거하세요.
  2) 배수 호스가 꺾이거나 막히지 않았는지 확인하세요.
  3) 물이 남아 있으면 응급 배수 호스를 사용하세요.
  - 위 조치 후에도 반복되면 서비스 점검이 필요할 수 있습니다.
[로봇청소기 사용자 가이드(더미)]
- 도킹이 안 될 때:
  1) 도킹 스테이션 앞 1m 이내 장애물을 제거하세요.
  2) 충전 단자에 먼지가 있으면 마른 천으로 닦으세요.
  3) 지도/맵이 손상되었을 수 있으니 '맵 재설정'을 고려하세요(초기화 아님).
- 충돌(bump)이 많을 때:
  - 조명/거울/유광 바닥에서 센서 오인식이 발생할 수 있습니다.


## 선택 과제 
> **풀이 순서**: 먼저 아래 `✅ TODO 셀`의 주석 1~3번을 푸세요. 이 마크다운의 문제 1~3은 **심화 과제**이며, 풀이는 `✅ 참고 답안` 셀에 함께 들어 있습니다.
### 문제 1) 새 문서 추가 자동 반영 (중)
- `manual_remote.txt`(재택근무 매뉴얼)를 새로 만들고,
- 위 Dynamic Tool Generation 코드 수정 없이(또는 최소 수정으로) 자동으로 tool이 생성되게 하세요.

### 문제 2) Citation 강제 포맷 강화 (상)
- 답변을 아래 포맷으로 강제해보세요.
  - 항목별로: `- 결론 ... (근거: [category:chunkN])`
- Self-check에서 포맷이 깨지면 재작성하도록 강화하세요.

### 문제 3) 간단 평가셋 만들기 (상)
- 질문 8개 정도를 만들고,
- 각 질문에 대해 "필수로 호출돼야 하는 카테고리"를 적어둔 뒤,
- agent 실행 로그(messages)를 분석해 tool 호출이 맞는지 점검하는 함수를 작성해보세요.

> 아래 셀은 학생용 TODO 입니다.


In [20]:
# ✅ TODO 셀 (학생용)
# 1) 새로운 매뉴얼 파일을 추가해보세요. (예: manual_air_purifier.txt)
# 2) 파일을 추가한 뒤, manual_files/retrievers/tools를 재생성하여 자동 반영되는지 확인하세요.
# 3) Self-check를 강화해보세요:
#    - 답변에 출처 태그가 '2개 이상' 없으면 수정 요청
#    - "매뉴얼에 없음"이라고 말할 때도, 검색 결과가 비어있었음을 근거로 남기기

raise NotImplementedError("학생용 TODO 셀입니다. 아래 '참고 답안'을 보기 전까지 구현해보세요.")


NotImplementedError: 학생용 TODO 셀입니다. 아래 '참고 답안'을 보기 전까지 구현해보세요.

In [ ]:
# ✅ 참고 답안 (핵심만) — 막힐 때만 참고하세요

# --- 문제 1) 새 문서 추가 → 코드 수정 없이 자동 반영 ---
# 핵심: manual_*.txt 규칙으로 저장한 뒤, glob 스캔 + retrievers/tools 재생성만 하면 됩니다.
with open("manual_air_purifier.txt", "w", encoding="utf-8") as f:
    f.write("[공기청정기 가이드(더미)]\n- 필터 경고가 뜨면 교체 권장.\n- 냄새가 나면 환기 후 필터 상태 확인.\n")

manual_files = sorted(glob.glob("manual_*.txt"))
retrievers = {
    os.path.splitext(os.path.basename(f))[0].replace("manual_", ""): build_retrievers_for_file(f)
    for f in manual_files
}
tools = [make_search_tool(cat) for cat in retrievers.keys()]
print("새 카테고리:", list(retrievers.keys()))

# --- 문제 2) Citation 강제 포맷 + Self-check 강화 ---
SYSTEM_V2 = SYSTEM + """
- 각 항목은 반드시 '- 결론 ... (근거: [category:chunkN])' 형식으로 작성.
- 출처 태그는 최소 2개 이상.
- '매뉴얼/가이드에 없음'이라고 답할 땐 검색 결과가 비어있었음을 함께 근거로 설명.
"""
agent = create_agent(llm, tools, system_prompt=SYSTEM_V2)

def answer_with_self_check_v2(question: str) -> str:
    draft = agent.invoke({"messages": [("user", question)]})["messages"][-1].content
    # 포맷/출처가 부족하면 1회 재작성 요청
    if ("(근거: [" not in draft) or (draft.count(":chunk") < 2):
        fix = f"아래 답변을 '- 결론 ...(근거: [category:chunkN])' 형식 + 출처 2개 이상으로 다시 작성.\n질문:{question}\n초안:{draft}"
        draft = agent.invoke({"messages": [("user", fix)]})["messages"][-1].content
    return draft

# --- 문제 3) 간단 평가셋: 필수 카테고리 도구가 실제로 호출됐는지 점검 ---
EVAL = [
    {"q": "세탁기 5C 에러 해결법 알려줘", "must": "washing_machine"},
    {"q": "로봇청소기 도킹이 안돼요",     "must": "robot_vacuum"},
    {"q": "에어컨 필터 경고 떴어요",      "must": "air_conditioner"},
]
def check_tool_calls(item):
    res = agent.invoke({"messages": [("user", item["q"])]})
    called = [getattr(m, "name", None) for m in res["messages"] if getattr(m, "name", None)]
    return {"q": item["q"], "must": item["must"], "called": called,
            "ok": any(item["must"] in (c or "") for c in called)}
# for it in EVAL: print(check_tool_calls(it))

## ✅ 정리 & 스스로 점검

**정리**
- RAG = **검색(근거 찾기) + 생성(근거로 답하기)**. 모델을 다시 학습하지 않고도 새 지식을 쓸 수 있습니다.
- 의미 검색(임베딩)과 키워드 검색(BM25)을 섞으면 서로의 약점을 보완합니다.
- 출처 태그(`[category:chunkN]`)로 답의 근거를 추적할 수 있게 만듭니다.

**RAG 품질을 올리는 팁**
| 문제 | 시도해 볼 것 |
|---|---|
| 엉뚱한 조각이 검색됨 | chunk_size 조정, k 조정, 하이브리드 가중치 조정 |
| 근거가 있는데 답이 틀림 | 시스템 프롬프트에 "근거에 없는 내용은 말하지 말 것" 강조 |
| 질문이 여러 기기에 걸침 | "필요한 도구를 모두 호출" 규칙 + 평가셋으로 확인 (문제 3) |

**스스로 점검**
1. 에러코드 "5C" 검색에는 임베딩과 BM25 중 어느 쪽이 더 믿을 만할까요? 왜일까요?
2. 매뉴얼에 없는 질문을 하면 에이전트는 어떻게 답해야 할까요?
3. RAG와 모델 미세조정은 각각 언제 쓰는 게 좋을까요?

**다음 단계 →** 08번: 여러 에이전트가 역할을 나눠 데이터 분석 보고서를 만듭니다.